# MTLnet — Colab Training

Multi-task learning for POI prediction (category + next-POI).

**Typical workflow:**
1. Run **Setup** (sections 1–5) once per session
2. Set your **Pipeline Configuration** (section 6)
3. Run **Step 1** → Generate Embeddings
4. Run **Step 2** → Create Model Inputs
5. Run **Step 3** → Train

**Drive layout expected:**
```
MyDrive/mestrado/PoiMtlNet/
  data/checkins/      ← raw checkins (Florida.parquet, Alabama.parquet, …)
  data/miscellaneous/ ← shapefiles (tl_2022_*_tract_*/)
  output/             ← generated embeddings and model inputs  [auto-created]
  results/            ← training results                        [auto-created]
```

---
## ① Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/mestrado/PoiMtlNet')
DATA_ROOT    = DRIVE_ROOT / 'data'
OUTPUT_DIR   = DRIVE_ROOT / 'output'
RESULTS_ROOT = DRIVE_ROOT / 'results'

for p in [DATA_ROOT, OUTPUT_DIR, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['DATA_ROOT']    = str(DATA_ROOT)
os.environ['OUTPUT_DIR']   = str(OUTPUT_DIR)
os.environ['RESULTS_ROOT'] = str(RESULTS_ROOT)

print(f'DATA_ROOT:    {DATA_ROOT}  (exists={DATA_ROOT.exists()})')
print(f'OUTPUT_DIR:   {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})')
print(f'RESULTS_ROOT: {RESULTS_ROOT}  (exists={RESULTS_ROOT.exists()})')

DATA_ROOT:    /content/drive/MyDrive/mestrado/PoiMtlNet/data  (exists=True)
OUTPUT_DIR:   /content/drive/MyDrive/mestrado/PoiMtlNet/output  (exists=True)
RESULTS_ROOT: /content/drive/MyDrive/mestrado/PoiMtlNet/results  (exists=True)


In [3]:
REPO_DIR = Path('/content/PoiMtlNet')

if not REPO_DIR.exists():
    !git clone https://github.com/VitorHugoOli/PoiMtlNet.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git pull
!git log --oneline -3

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 438 bytes | 438.00 KiB/s, done.
From https://github.com/VitorHugoOli/PoiMtlNet
   10d6b95..327bee9  main       -> origin/main
Updating 10d6b95..327bee9
Fast-forward
 src/utils/mps.py | 6 ++++--
 1 file changed, 4 insertions(+), 2 deletions(-)
/content/PoiMtlNet
Already up to date.
327bee9 (HEAD -> main, origin/main, origin/HEAD) feat: some changes
10d6b95 Merge remote-tracking branch 'origin/main'
3bec993 feat: some changes


In [8]:
# Base requirements
!pip install -q -r requirements_colab.txt

# NashMTL requires cvxpy + ECOS solver
!pip install -q cvxpy

# PyTorch Geometric — needed for HGI, DGI, POI2HGI, Check2HGI
!pip install -q torch-geometric

!pip install -q pytorch_warmup

!pip install -q torchmetrics

import torch

!pip uninstall torch-scatter torch-sparse torch-geometric torch-cluster  --y
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install git+https://github.com/pyg-team/pytorch_geometric.git
!pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.5 MB/s eta 0:00:00
Found existing installation: torch_scatter 2.1.2+pt26cu124
Uninstalling torch_scatter-2.1.2+pt26cu124:
  Successfully uninstalled torch_scatter-2.1.2+pt26cu124
Found existing installation: torch_sparse 0.6.18+pt26cu124
Uninstalling torch_sparse-0.6.18+pt26cu124:
  Successfully uninstalled torch_sparse-0.6.18+pt26cu124
Found existing installation: torch-geometric 2.8.0
Uninstalling torch-geometric-2.8.0:
  Successfully uninstalled torch-geometric-2.8.0
Found existing installation: torch_cluster 1.6.3+pt26cu124
Uninstalling torch_cluster-1.6.3+pt26cu124:
  Successfully uninstalled torch_cluster-1.6.3+pt26cu124
Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html
  Using cached https://data.pyg.org/whl/torch-2.6.0%2Bcu124/torch_scatter-2.1.2%2Bpt26cu124-cp312-cp312-linux_x86_64.whl (10.8 MB)
Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html
  Using cached https://data.pyg.org/whl/

In [4]:
import sys

for sub in ('src', 'research'):
    p = str(REPO_DIR / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

from configs.globals import DEVICE
print('Device:', DEVICE)

Device: cuda


---
## ② Pipeline Configuration

**Edit this cell before running any pipeline step.**  
All subsequent cells read `STATE`, `ENGINE`, and `TASK` from here.

In [5]:
# ── edit here ─────────────────────────────────────────────────────────────────
STATE  = 'texas'   # florida | alabama | georgia | california | texas | arizona
ENGINE = 'fusion'       # hgi | dgi | poi2hgi | check2hgi | time2vec | space2vec | sphere2vec | fusion
TASK   = 'mtl'       # mtl | category | next

# Optional training overrides (None = use config defaults)
EPOCHS = None        # e.g. 50
FOLDS  = None        # e.g. 1 for a quick smoke test
# ──────────────────────────────────────────────────────────────────────────────

from configs.paths import EmbeddingEngine, IoPaths, Resources
ENGINE_ENUM = EmbeddingEngine(ENGINE)

# Map state name to shapefile resource (needed by graph-based engines)
_SHAPEFILES = {
    'florida':    Resources.TL_FL,
    'alabama':    Resources.TL_AL,
    'georgia':    Resources.TL_GA,
    'california': Resources.TL_CA,
    'texas':      Resources.TL_TX,
    'arizona':    Resources.TL_AZ,
}
SHAPEFILE = _SHAPEFILES.get(STATE.lower())

print(f'State:     {STATE}')
print(f'Engine:    {ENGINE}')
print(f'Task:      {TASK}')
print(f'Shapefile: {SHAPEFILE}')
print(f'Epochs:    {EPOCHS or "(default)"}')
print(f'Folds:     {FOLDS  or "(all)"}')

State:     texas
Engine:    fusion
Task:      mtl
Shapefile: /content/drive/MyDrive/mestrado/PoiMtlNet/data/miscellaneous/tl_2022_48_tract_TX/tl_2022_48_tract.shp
Epochs:    (default)
Folds:     (all)


---
## ③ Step 1 — Generate Embeddings

Run the subsection that matches your `ENGINE`.  
Skip this step entirely if embeddings are already in `output/` on Drive.

### HGI / POI2HGI / Check2HGI  (`engine = hgi | poi2hgi | check2hgi`)

In [9]:
# Run only if ENGINE in ('hgi', 'poi2hgi', 'check2hgi')
assert ENGINE in ('hgi', 'poi2hgi', 'check2hgi'), f'Wrong engine: {ENGINE}'
assert SHAPEFILE is not None, f'No shapefile registered for state "{STATE}"'

import importlib.util
from copy import copy

# Load the correct pipeline module without executing __main__
_pipe_path = REPO_DIR / f'pipelines/embedding/{ENGINE}.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

# Patch STATES to run only our chosen state
_pipe.STATES = {
    STATE.capitalize(): {'shapefile': SHAPEFILE},
}

_pipe.run_pipeline()

  Cross-region edge weight w_r = 0.7
Reading POI data...
Reading boroughs data...
  Encoded 7 categories: ['Community', 'Entertainment', 'Food', 'Nightlife', 'Outdoors']...
  Encoded 365 fclasses: ['24 Hour Fitness', 'AMC Theatre', 'AT&T', 'Accessories', 'Administration']...
Creating spatial graph...
Computing region features...
Loading POI embeddings...
  Saved encodings to /content/drive/MyDrive/mestrado/PoiMtlNet/output/hgi/texas/temp/encodings.json
✓ Phase 3a complete: Delaunay graph created
  POIs: 160938, Regions: 6553, Edges: 482789
  Intermediate files saved to: /content/drive/MyDrive/mestrado/PoiMtlNet/output/hgi/texas/temp
    - edges.csv (Delaunay graph)
    - pois.csv (POI-region mapping)
    - encodings.json (category/fclass mappings)
POI2Vec Training: Texas
Loading edges and POIs...
Initializing Node2Vec walk generator...
  POIs: 160938
  Edges: 482789
  Unique fclass values: 365
PHASE 3b: Generating Random Walks
Generating walks (batch_size=128)...


Walk generation: 100%|██████████| 1258/1258 [07:28<00:00,  2.81it/s]


Generated 4828140 fclass walks
Building global co-occurrence statistics...
Co-occurrence stats computed for 365 fclass values

PHASE 3c: Training fclass Embeddings
Extracting category-fclass hierarchy pairs...
  Found 365 unique (category, fclass) pairs
Building POISet dataset...
Initializing EmbeddingModel...
Training for 100 epochs (batch=2048, lr=0.05, workers=10, device=cuda:None)...


Epoch 1/100: 100%|██████████| 2358/2358 [00:34<00:00, 67.38it/s, loss=0.0533, h_loss=5.49e-04]


  Epoch 1/100: Loss=0.1526, Hierarchy Loss=3.07e-04


...

  Epoch 100/100: Loss=0.0360, Hierarchy Loss=1.63e-02
Extracting fclass embeddings...
  Shape: (365, 64) (best epoch loss=0.0348)

PHASE 3d: POI-Level Embedding Reconstruction
Mapping each POI to its fclass embedding...
Note: Multiple POIs with same fclass will share the same embedding

  ✓ Mapped 160938 POIs to their fclass embeddings
  Average POIs per fclass: 440.9
  Max POIs sharing same fclass: 9192
  Min POIs per fclass: 1

Saved fclass embeddings: /content/drive/MyDrive/mestrado/PoiMtlNet/output/hgi/texas/poi2vec_fclass_embeddings_Texas.pt
  Shape: (365, 64)
Saved POI embeddings (CSV): /content/drive/MyDrive/mestrado/PoiMtlNet/output/hgi/texas/poi2vec_poi_embeddings_Texas.csv
  Columns: ['placeid', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '

Training HGI: 100%|██████████| 2000/2000 [34:06<00:00,  1.02s/it, best=0.4314, loss=0.4378]


POI embeddings: /content/drive/MyDrive/mestrado/PoiMtlNet/output/hgi/texas/embeddings.parquet (160938, 64)
Region embeddings: /content/drive/MyDrive/mestrado/PoiMtlNet/output/hgi/texas/region_embeddings.parquet (6553, 64)


Processing batches: 100%|██████████| 5/5 [03:23<00:00, 40.79s/it]


{'Texas': True}

### DGI  (`engine = dgi`)

In [ ]:
assert ENGINE == 'dgi', f'Wrong engine: {ENGINE}'
assert SHAPEFILE is not None, f'No shapefile registered for state "{STATE}"'

import importlib.util

_pipe_path = REPO_DIR / 'pipelines/embedding/dgi.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {
    STATE.capitalize(): {'shapefile': SHAPEFILE},
}

_pipe.run_pipeline()

### Time2Vec  (`engine = time2vec`)

In [12]:
assert ENGINE == 'time2vec', f'Wrong engine: {ENGINE}'

import importlib.util

_pipe_path = REPO_DIR / 'pipelines/embedding/time2vec.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {STATE.capitalize(): {}}

_pipe.run_pipeline()

Loading check-ins from: /content/drive/MyDrive/mestrado/PoiMtlNet/data/checkins/Texas.parquet
Loaded 4089892 check-ins
Extracting temporal features...
Valid check-ins: 4089892
Time features shape: (4089892, 2)
Initializing model: activation=sin, out_features=64, embed_dim=64
Creating contrastive dataset (mode=feat_space)...
Total pairs generated: 1950858 (mode=feat_space)
Materialising pair tensors...
  feat_i: (1950858, 2)  memory: 39.0MB
Compiling model with torch.compile(mode='default') ...
Training for 100 epochs on cuda...


Training:   0%|          | 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:194: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
Training: 100%|██████████| 100/100 [04:27<00:00,  2.67s/it, loss=0.1060]


Model saved to: /content/drive/MyDrive/mestrado/PoiMtlNet/output/time2vec/texas/time2vec_model.pt
Generating embeddings...
Embeddings saved to: /content/drive/MyDrive/mestrado/PoiMtlNet/output/time2vec/texas/embeddings.parquet
Embedding dimensions: (4089892, 64)


Processing users: 100%|██████████| 38644/38644 [03:06<00:00, 207.37it/s]


{'Texas': True}

### Space2Vec / Sphere2Vec  (`engine = space2vec | sphere2vec`)

In [15]:
assert ENGINE in ('sphere2vec'), f'Wrong engine: {ENGINE}'

import importlib.util

_pipe_path = REPO_DIR / f'pipelines/embedding/{ENGINE}.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {STATE.capitalize(): {}}

_pipe.run_pipeline()

Loading check-ins from: /content/drive/MyDrive/mestrado/PoiMtlNet/data/checkins/Texas.parquet
Loaded 4089892 check-ins
Coordinates shape: (4089892, 2)
Total batches per epoch: 998
Encoder variant: paper
Training for 50 epochs on cuda...


Training: 100%|██████████| 50/50 [06:04<00:00,  7.29s/it, loss=0.7747]


Model saved to: /content/drive/MyDrive/mestrado/PoiMtlNet/output/sphere2vec/texas/sphere2vec_model.pt
Generating embeddings (train mode, dropout active — matches notebook)...


Embedding: 100%|██████████| 409/409 [00:00<00:00, 496.56it/s]


Embeddings saved to: /content/drive/MyDrive/mestrado/PoiMtlNet/output/sphere2vec/texas/embeddings.parquet
Per-checkin embeddings: (4089892, 66)
Per-POI embeddings:     (160938, 66)


Processing batches: 100%|██████████| 5/5 [01:32<00:00, 18.48s/it]


{'Texas': True}

### Fusion  (`engine = fusion`)

Fusion concatenates multiple pre-existing embeddings — run after each component engine is done.

In [8]:
# assert ENGINE == 'fusion', f'Wrong engine: {ENGINE}'

import importlib.util

_pipe_path = REPO_DIR / 'pipelines/fusion.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {STATE.capitalize(): {}}

_pipe.run_pipeline()


=== Generating Category Input ===
Embeddings: ['sphere2vec', 'hgi']
Total dimension: 128
Base POIs: 160938
Aligning embeddings...
Fusing embeddings...
✓ Category input saved: /content/drive/MyDrive/mestrado/PoiMtlNet/output/fusion/texas/input/category.parquet
  Shape: (160938, 130)
  Columns: ['placeid', 'category', '0', '1', '2']... ['126', '127']

=== Generating Next-POI Input ===
Embeddings: ['hgi', 'time2vec']
Total dimension: 128
Loading check-ins...
Check-ins: 4089892
Aligning embeddings...
Fusing embeddings...
Using check-in-level conversion (position-based lookup)...


Processing users: 100%|██████████| 38644/38644 [05:51<00:00, 110.07it/s]


Sequences saved: 460976 sequences → /content/drive/MyDrive/mestrado/PoiMtlNet/output/fusion/texas/input/temp/sequences_next.parquet
✓ Next-POI input saved (check-in level): /content/drive/MyDrive/mestrado/PoiMtlNet/output/fusion/texas/input/next.parquet


{'Texas': True}

---
## ④ Step 2 — Create Model Inputs

Generates `category.parquet` and `next.parquet` inside `output/{engine}/{state}/input/`.
Skip if inputs are already present.

In [ ]:
import importlib.util

_pipe_path = REPO_DIR / 'pipelines/create_inputs.pipe.py'
_spec = importlib.util.spec_from_file_location('_pipe', _pipe_path)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

_pipe.STATES = {
    STATE.capitalize(): {
        'engine': ENGINE_ENUM,
        'use_checkin': ENGINE == 'time2vec',  # time2vec uses check-in-level embeddings
    }
}

_pipe.run_pipeline()

In [ ]:
# Verify inputs exist before training
cat_path  = IoPaths.get_category(STATE, ENGINE_ENUM)
next_path = IoPaths.get_next(STATE, ENGINE_ENUM)

print(f'Category input  ({"OK" if cat_path.exists() else "MISSING"}): {cat_path}')
print(f'Next-POI input  ({"OK" if next_path.exists() else "MISSING"}): {next_path}')

---
## ⑤ Step 3 — Train

Results (metrics CSVs, classification reports, plots, checkpoints) are written to Drive under `results/`.

In [6]:
# Build the training command from the config set in section ②
_cmd = f'python scripts/train.py --task {TASK} --state {STATE} --engine {ENGINE}'
if EPOCHS is not None:
    _cmd += f' --epochs {EPOCHS}'
if FOLDS is not None:
    _cmd += f' --folds {FOLDS}'

print('Command:', _cmd)

Command: python scripts/train.py --task mtl --state texas --engine fusion


In [ ]:
# Full run
!{_cmd}

In [7]:
# Smoke test — 1 fold, 5 epochs (confirm setup is correct before the full run)
!python scripts/train.py --task {TASK} --state {STATE} --engine {ENGINE} --folds 1 --epochs 5

2026-04-13 03:24:50,781 - INFO - Creating 2-fold CV for mtl
2026-04-13 03:24:50,781 - INFO - Loading next-POI data: texas/fusion
2026-04-13 03:27:07,651 - INFO - Next-POI data: (460976, 1152) (dim=128, window=9), distribution: {np.int64(0): np.int64(70084), np.int64(1): np.int64(49025), np.int64(2): np.int64(142804), np.int64(3): np.int64(29576), np.int64(4): np.int64(28654), np.int64(5): np.int64(110090), np.int64(6): np.int64(30743)}
2026-04-13 03:27:12,062 - INFO - Loading category data: texas/fusion
2026-04-13 03:27:12,337 - INFO - Category data: (160938, 128) (dim=128), distribution: {np.int64(0): np.int64(29886), np.int64(1): np.int64(15955), np.int64(2): np.int64(38039), np.int64(3): np.int64(4956), np.int64(4): np.int64(12125), np.int64(5): np.int64(51338), np.int64(6): np.int64(8639)}
2026-04-13 03:27:20,464 - INFO - POI→users mapping: 160938 POIs → /content/drive/MyDrive/mestrado/PoiMtlNet/output/fusion/texas/input/poi_user_mapping.json
2026-04-13 03:27:39,453 - INFO - Sequen

---
## ⑥ Inspect Results

In [ ]:
import pandas as pd

results_dir = IoPaths.get_results_dir(STATE, ENGINE_ENUM)
print('Results directory:', results_dir)
print()

csv_files = sorted(results_dir.glob('*.csv'))
if csv_files:
    for f in csv_files:
        print(f'--- {f.name} ---')
        print(pd.read_csv(f).to_string(index=False))
        print()
else:
    print('No CSV results found yet.')